In [22]:
Sys.setenv(RETICULATE_PYTHON = "/data1/lesliec/tyler/utils/miniforge3/envs/multiome/bin/python")
setwd("~/0-workspace/CCR7_DC/GSE278161-sawa-2026-tcp/")

suppressPackageStartupMessages({
    library(curl)
    library(Seurat) # v4.3.0.1
    library(Rmagic)
    library(Matrix)
    library(future)
    library(dplyr)
    library(ggplot2)
    library(cowplot)
    library(ComplexHeatmap)
    library(circlize)
    library(ggrepel)
    library(anndata)
})

set.seed(1)
options(future.globals.maxSize = Inf)
source("../utils.R")

In [23]:
pal <- list(
  Clusters = c(
    "#00bf00", "#489de8", "#d40663", "#f8c72f", "#077315",
    "#785cd4", "#e67109", "#0eefff", "#f081e6", "#260691",
    "#49709c", "#9e7d3f", "#bd537a", "#4e225c", "#f202ed",
    "#fec55f", "#062e0b", "#9c6fa8", "#078d94", "#5c1a1a",
    "#827c68", "#aebeff", "#9c2903", "#ffc5af", "#4f5715",
    "#0249f0", "#f43525", "#0077ff", "#7f227e", "#dfddff",
    "#7e85d7", "#fff64f", "#5fed0e", "#543018", "#f31220"
  ),
  Clusters_long = c(
    "#000000", "#FAD09F", "#1CE6FF", "#FF34FF", "#FF4A46", "#008941", "#006FA6", "#A30059",
    "#FFDBE5", "#7A4900", "#0000A6", "#63FFAC", "#B79762", "#004D43", "#8FB0FF", "#997D87",
    "#5A0007", "#809693", "#1B4400", "#4FC601", "#3B5DFF", "#4A3B53", "#FF2F80",
    "#61615A", "#BA0900", "#6B7900", "#00C2A0", "#FFAA92", "#FF90C9", "#B903AA", "#D16100",
    "#DDEFFF", "#000035", "#7B4F4B", "#A1C299", "#300018", "#0AA6D8", "#013349", "#00846F",
    "#372101", "#FFFF00", "#C2FFED", "#A079BF", "#CC0744", "#C0B9B2", "#C2FF99", "#001E09",
    "#00489C", "#6F0062", "#0CBD66", "#EEC3FF", "#456D75", "#B77B68", "#7A87A1", "#788D66",
    "#885578", "#FAD09F", "#FF8A9A", "#D157A0", "#BEC459", "#456648", "#0086ED", "#886F4C",
    "#34362D", "#B4A8BD", "#00A6AA", "#452C2C", "#636375", "#A3C8C9", "#FF913F", "#938A81",
    "#575329", "#00FECF", "#B05B6F", "#8CD0FF", "#3B9700", "#04F757", "#C8A1A1", "#1E6E00",
    "#7900D7", "#A77500", "#6367A9", "#A05837", "#6B002C", "#772600", "#D790FF", "#9B9700",
    "#549E79", "#FFF69F", "#201625", "#72418F", "#BC23FF", "#99ADC0", "#3A2465", "#922329",
    "#5B4534", "#FDE8DC", "#404E55", "#0089A3", "#CB7E98", "#A4E804", "#324E72", "#6A3A4C")
)

dir.create('plots')
saveRDS(pal, "plots/palette.rds")
pal <- readRDS("plots/palette.rds")

Warning message in dir.create("plots"):
“'plots' already exists”


# load data

In [ ]:
# load data
data.path <- '/data1/lesliec/tyler/data/GSE278161-sawa-2026-tcp/data/'
counts1 <- ReadMtx(
    mtx = file.path(data.path, "GSM8541018_RORgt_high_matrix.mtx.gz"),
    features = file.path(data.path, "GSM8541018_RORgt_high_features.tsv.gz"),
    cells = file.path(data.path, "GSM8541018_RORgt_high_barcodes.tsv.gz"),
)

counts2 <- ReadMtx(
    mtx = file.path(data.path, "GSM8541019_LTi_matrix.mtx.gz"),
    features = file.path(data.path, "GSM8541019_LTi_features.tsv.gz"),
    cells = file.path(data.path, "GSM8541019_LTi_barcodes.tsv.gz"),
)

In [ ]:
gene.info <- read.table(file.path(data.path, 'GSM8541018_RORgt_high_features.tsv.gz'), sep='\t', header=F)[,1:2]
colnames(gene.info) <- c("ID", "symbol")
gene.info$symbol.unique <- make.unique(gene.info$symbol)
rownames(gene.info) <- gene.info$symbol.unique

In [ ]:
sro1 <- CreateSeuratObject(
  project = "RORgtHigh",
  counts = counts1
)
sro2 <- CreateSeuratObject(
  project = "LTi",
  counts = counts2
)

In [ ]:
sro <- merge(sro1, sro2, add.cell.ids = c("RORgtHigh", "LTi"))

In [ ]:
sro <- PercentageFeatureSet(sro, pattern = "^mt-", col.name = "MtFrac_RNA")
sro$sample <- sro$orig.ident
sro@assays$RNA <- AddMetaData(sro@assays$RNA, gene.info[rownames(sro), ])

In [ ]:
# barcodes with < 500 gene count (nFeature_RNA) or > 5% proportion of mitochondrial genes (percent.mt) 

In [ ]:
thr <- data.frame(#nc.min = 1000, 
                  nf.min = 500, 
                  mp.max = 5)

In [ ]:
dir.create('plots/QC')
pdf("plots/QC/violin-QC-sample.pdf", width = 8, height = 6)
plot.all.QC(sro, ident = "sample", thr = thr)
dev.off()

In [ ]:
cell.discard <- #sro$nCount_RNA < thr$nc.min |
  sro$nFeature_RNA < thr$nf.min | 
  sro$MtFrac_RNA > thr$mp.max
table(cell.discard)

In [ ]:
res.list <- seq(0.2, 2, 0.2)
sro <- sro[, !cell.discard] # not run here

# cluster and umap ####
cell.count <- rowSums(sro@assays$RNA@counts > 0)
sro <- NormalizeData(sro[cell.count > 1, ]) %>%
  FindVariableFeatures(selection.method = "vst", nfeatures = 2000)

sro <- ScaleData(sro) %>%
  RunPCA(npcs = 50) %>%
  FindNeighbors(dims = 1:30, k.param = 30) %>%
  FindClusters(resolution = res.list) %>%
  RunUMAP(dims = 1:30, n.neighbors = 30, metric = "cosine", min.dist = 0.4, spread = 1)

In [ ]:
s.genes <- rownames(sro)[toupper(rownames(sro)) %in% toupper(cc.genes.updated.2019$s.genes)]
g2m.genes <- rownames(sro)[toupper(rownames(sro)) %in% toupper(cc.genes.updated.2019$g2m.genes)]
sro <- CellCycleScoring(sro, s.features = s.genes, g2m.features = g2m.genes, set.ident = F)
sro$Phase <- gsub("G2M", "G2/M", sro$Phase)

In [ ]:
dir.create("Seurat")

In [ ]:
## cellxgene ####
unimp.expr.dgC <- t(sro@assays$RNA$data)
ad <- AnnData(
    X = unimp.expr.dgC,
    obs = sro@meta.data,
    obsm = list(
        X_umap = sro@reductions$umap@cell.embeddings
    )
)
write_h5ad(ad, paste0("Seurat/unimputed-expr.h5ad"))

In [ ]:
# save results ####
dir.create("Seurat")
write.csv(sro@meta.data, file = "Seurat/meta-data.csv", quote = F)
write.csv(sro@reductions$pca@cell.embeddings, file = "Seurat/PCA.csv", quote = F)
write.csv(sro@reductions$umap@cell.embeddings, file = "Seurat/UMAP.csv", quote = F)
saveRDS(sro, file = "Seurat/SRO.rds")

In [ ]:
sro.imp <- magic(sro)
saveRDS(sro.imp@assays$MAGIC_RNA@data, file = "Seurat/imputed-expr.rds")

# plots

In [24]:
sro <- readRDS("Seurat/SRO.rds")

In [ ]:
for (res in res.list){
    group.name <- paste0("RNA_snn_res.", res)
    pl <- plot.all.QC(sro, ident = group.name, CC = T, col = pal$Clusters_long)
    pdf(paste0("plots/QC/vln-QC-", group.name, "-QC.pdf"), width = 10, height = 10)
    print(pl)
    dev.off()
}

In [5]:
pref.p.sro <- "plots/"

In [ ]:
pdf(paste0(pref.p.sro, "QC/UMAP-QC.pdf"), width = 8, height = 6)
plot.continuous.value(sro, idx = rownames(sro@meta.data), vis = sro@reductions$umap@cell.embeddings,
                      val = sro$nCount_RNA, val.name='nCount_RNA', point.size=1.5)
plot.continuous.value(sro, idx = rownames(sro@meta.data), vis = sro@reductions$umap@cell.embeddings,
                      val = sro$nFeature_RNA, val.name='nFeature_RNA', point.size=1.5)
plot.continuous.value(sro, idx = rownames(sro@meta.data), vis = sro@reductions$umap@cell.embeddings,
                      val = sro$MtFrac_RNA, val.name='MtFrac_RNA', point.size=1.5)
plot.continuous.value(sro, idx = rownames(sro@meta.data), vis = sro@reductions$umap@cell.embeddings,
                      val = sro$S.Score, val.name='S.Score', point.size=1.5)
plot.continuous.value(sro, idx = rownames(sro@meta.data), vis = sro@reductions$umap@cell.embeddings,
                      val = sro$G2M.Score, val.name='G2M.Score', point.size=1.5)
dev.off()

In [ ]:
pdf(paste0(pref.p.sro, "UMAP-sample.pdf"), width = 8, height = 6)
plot.clusters(sro, groups = sro$sample, clusters.col = "sample",
              col = pal$Clusters, label.size = 3, labels = F, point.size = 1.5,
              label.pad = 1, pref.C = F)
dev.off()

In [7]:
res.list <- seq(0.2, 2, 0.2)

In [9]:
pdf(paste0(pref.p.sro, "UMAP-clusters.pdf"), width = 8, height = 6)
for (res in res.list){
    colname <- paste0("RNA_snn_res.", res)
    print(
        plot.clusters(sro, groups = sro@meta.data[[colname]],
                      clusters.col = colname, col = pal$Clusters,
                      label.size = 3, point.size = 1.5,
                      pref.C = T, raster=T)
    )
}
dev.off()

pdf 
  2

# annotate and recluster

In [3]:
sro <- readRDS("Seurat/SRO.rds")

In [4]:
sro.sub <- subset(sro, RNA_snn_res.1 %ni% c(1, 5, 13, 14, 15))

In [7]:
table(sro.sub$`RNA_snn_res.1`)


  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16 
561   0 528 445 412   0 301 279 267 239 198 185 161   0   0   0  17 

In [8]:
res.list <- seq(0.2, 2, 0.2)

# cluster and umap ####
cell.count <- rowSums(sro.sub@assays$RNA@counts > 0)
sro.sub <- NormalizeData(sro.sub[cell.count > 1, ]) %>%
  FindVariableFeatures(selection.method = "vst", nfeatures = 2000)

sro.sub <- ScaleData(sro.sub) %>%
  RunPCA(npcs = 50) %>%
  FindNeighbors(dims = 1:30, k.param = 30) %>%
  FindClusters(resolution = res.list) %>%
  RunUMAP(dims = 1:30, n.neighbors = 30, metric = "cosine", min.dist = 0.4, spread = 1)

Centering and scaling data matrix

PC_ 1 
Positive:  Tmem176b, Ftl1, Bhlhe40, Cd74, Rgs2, Dennd4a, Junb, Malat1, Cd7, Ctsl 
	   Nebl, Ccr6, H2-Eb1, Ctss, Ifrd1, Asb2, H2-Aa, Sat1, H2-Ab1, Lst1 
	   Klf4, Atf3, Itgav, Ttn, Ppp1r15a, Dscam, Mndal, Tiparp, Nfkbia, Ahcyl2 
Negative:  Pclaf, Top2a, Stmn1, Mki67, Cdk1, Spc24, Birc5, Cdca8, Hmgb2, Nusap1 
	   Tuba1b, Hist1h2ap, Prc1, Kif11, Smc2, Aurkb, Tpx2, Cks1b, H2afx, Hist1h2ae 
	   Kif15, Cdca3, Hist1h1b, Tubb4b, Ccna2, Pbk, Ptma, Ube2c, Ran, Hist1h3c 
PC_ 2 
Positive:  Trbc2, Gem, Il7r, Camk4, Odc1, Hmgn3, Junb, Prickle1, Ifrd1, Ern1 
	   Tcf7, Itgav, Nfkbia, Itk, Dgat1, Dusp5, Vps37b, Nr4a2, Il23r, Il22 
	   Cdkn1a, Mapk10, Cebpb, Tnfaip3, Plekhg1, Ppp1r15a, Il2rb, Nrgn, Fos, Bhlhe40 
Negative:  Itgam, Spi1, Pld4, Cfp, Alox5ap, Rnase6, Kynu, Napsa, Hacd4, Marcks 
	   Wdfy4, Mpeg1, Cst3, Gm34680, Ly86, Hck, Gpx1, Rassf4, Kmo, Siglecg 
	   Cd180, Ctsh, Rogdi, Ffar2, Unc93b1, Skap2, Cyb561a3, H2-DMa, Cib1, Fgd2 
PC_ 3 
Positive:  H3f3b, 

Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 3593
Number of edges: 211612

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.9243
Number of communities: 4
Elapsed time: 0 seconds
Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 3593
Number of edges: 211612

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8819
Number of communities: 9
Elapsed time: 0 seconds
Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 3593
Number of edges: 211612

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8522
Number of communities: 10
Elapsed time: 0 seconds
Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 3593
Number of edges: 211612

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8285
Number of communities: 12
Elapsed time: 0 seconds
Modularity Optimiz

Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
16:47:09 UMAP embedding parameters a = 0.7669 b = 1.223

16:47:09 Read 3593 rows and found 30 numeric columns

16:47:09 Using Annoy for neighbor search, n_neighbors = 30

16:47:09 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

16:47:09 Writing NN index file to temp file /data1/lesliec/tyler/tmp/RtmpKE8AU6/file313fd441e5ce2

16:47:09 Searching Annoy index using 1 thread, search_k = 3000

16:47:10 Annoy recall = 100%

16:47:10 Commencing smooth kNN distance calibration using 1 thread
 with

In [10]:
# save results ####
pref <- 'Seurat/res1_ex_c1c5c13c14c15/'
dir.create(pref)
write.csv(sro.sub@meta.data, file = paste0(pref, "meta-data.csv"), quote = F)
write.csv(sro.sub@reductions$pca@cell.embeddings, file = paste0(pref, "PCA.csv"), quote = F)
write.csv(sro.sub@reductions$umap@cell.embeddings, file = paste0(pref, "UMAP.csv"), quote = F)
saveRDS(sro.sub, file = paste0(pref, "SRO.rds"))

Warning message in dir.create(pref):
“'Seurat/res1_ex_c1c5c13c14c15' already exists”


In [17]:
## cellxgene ####
unimp.expr.dgC <- t(sro.sub@assays$RNA$data)
ad <- AnnData(
    X = unimp.expr.dgC,
    obs = sro.sub@meta.data,
    obsm = list(
        X_umap = sro.sub@reductions$umap@cell.embeddings
    )
)
write_h5ad(ad, paste0("Seurat/res1_ex_c1c5c13c14c15/unimputed-expr.h5ad"))

In [11]:
pref.p.sro <- "plots/res1_ex_c1c5c13c14c15/"
dir.create(paste0(pref.p.sro, 'QC/'), recursive=T)

In [18]:
for (res in res.list){
    group.name <- paste0("RNA_snn_res.", res)
    pl <- plot.all.QC(sro.sub, ident = group.name, CC = T, col = pal$Clusters_long)
    pdf(paste0(pref.p.sro, "QC/vln-QC-", group.name, "-QC.pdf"), width = 10, height = 10)
    print(pl)
    dev.off()
}

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]

[[1]]

[[2]]

[[3]]

[[4]]

[[5]]

[[6]]

[[7]]



In [19]:
pdf(paste0(pref.p.sro, "QC/UMAP-QC.pdf"), width = 8, height = 6)
plot.continuous.value(sro.sub, idx = rownames(sro.sub@meta.data), vis = sro.sub@reductions$umap@cell.embeddings,
                      val = sro.sub$nCount_RNA, val.name='nCount_RNA', point.size=1.5)
plot.continuous.value(sro.sub, idx = rownames(sro.sub@meta.data), vis = sro.sub@reductions$umap@cell.embeddings,
                      val = sro.sub$nFeature_RNA, val.name='nFeature_RNA', point.size=1.5)
plot.continuous.value(sro.sub, idx = rownames(sro.sub@meta.data), vis = sro.sub@reductions$umap@cell.embeddings,
                      val = sro.sub$MtFrac_RNA, val.name='MtFrac_RNA', point.size=1.5)
plot.continuous.value(sro.sub, idx = rownames(sro.sub@meta.data), vis = sro.sub@reductions$umap@cell.embeddings,
                      val = sro.sub$S.Score, val.name='S.Score', point.size=1.5)
plot.continuous.value(sro.sub, idx = rownames(sro.sub@meta.data), vis = sro.sub@reductions$umap@cell.embeddings,
                      val = sro.sub$G2M.Score, val.name='G2M.Score', point.size=1.5)
dev.off()

pdf 
  2

In [20]:
pdf(paste0(pref.p.sro, "UMAP-sample.pdf"), width = 8, height = 6)
plot.clusters(sro.sub, groups = sro.sub$sample, clusters.col = "sample",
              col = pal$Clusters, label.size = 3, labels = F, point.size = 1.5,
              label.pad = 1, pref.C = F)
dev.off()

pdf 
  2

In [15]:
res.list <- seq(0.2, 2, 0.2)

In [21]:
pdf(paste0(pref.p.sro, "UMAP-clusters.pdf"), width = 8, height = 6)
for (res in res.list){
    colname <- paste0("RNA_snn_res.", res)
    print(
        plot.clusters(sro.sub, groups = sro.sub@meta.data[[colname]],
                      clusters.col = colname, col = pal$Clusters,
                      label.size = 3, point.size = 1.5,
                      pref.C = T, raster=T)
    )
}
dev.off()

pdf 
  2